<a href="https://colab.research.google.com/github/mmarguerittee/ML_projects/blob/main/%D0%BA%D0%BE%D0%BD%D1%82%D0%B5%D1%81%D1%822.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

In [ ]:
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

In [ ]:
test_id=test['id'].copy()
test.drop(['id'],axis=1,inplace=True)
X=train.copy()
y=train['target'].copy()
X.drop(['id','target'],axis=1,inplace=True)

In [ ]:
const_cols = [c for c in X.columns if X[c].nunique() == 1]
X = X.drop(columns=const_cols)
test = test.drop(columns=const_cols)

In [ ]:
cat_cols = [col for col in X.columns if "_cat" in col]

In [ ]:
X = X.drop(columns=[col for col in X.columns if "ps_calc" in col])
test = test.drop(columns=[col for col in test.columns if "ps_calc" in col])

In [ ]:
X, test = X.align(test, join="inner", axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
y.value_counts()

,count
target,
0,48178
1,1822


#обучение


In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=2000,
    depth=8,
    learning_rate=0.01,
    l2_leaf_reg=10,
    loss_function='Logloss',
    auto_class_weights='Balanced',
    verbose=200,
    random_seed=42
)

In [ ]:
model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100
)

0:	learn: 0.6926541	test: 0.6928507	best: 0.6928507 (0)	total: 220ms	remaining: 7m 20s
200:	learn: 0.6468446	test: 0.6733961	best: 0.6733961 (200)	total: 42.7s	remaining: 6m 21s
400:	learn: 0.6267507	test: 0.6713416	best: 0.6711841 (375)	total: 1m 23s	remaining: 5m 32s
600:	learn: 0.6118037	test: 0.6710356	best: 0.6707112 (540)	total: 2m	remaining: 4m 40s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6707112267
bestIteration = 540

Shrink model to first 541 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, iterations=2000, l2_leaf_reg=10, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=200)

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
best_f1 = 0
best_t = 0.5

for t in [i/1000 for i in range(10, 800)]:
    preds = (proba > t).astype(int)
    f1 = f1_score(y_test, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(best_t, best_f1)

0.59 0.116991643454039


In [ ]:
confusion_matrix(y_test, preds)

array([[9636,    0],
       [ 364,    0]])

In [ ]:
f1_score(y_test, preds)

0.0

In [ ]:
proba_test = model.predict_proba(test)[:, 1]
pr = (proba_test > best_t).astype(int)

In [ ]:
subm=pd.DataFrame({'id':test_id,'target':pr})
subm.to_csv('cc.csv',index=False)